# 06. Stochastic Rounding and Master-Weight-Free BF16 Training

- **Category:** 13_precision_and_numerics
- **Lineage & Citations:** Barnes et al. (1951, *Probabilistic Arithmetic*); Gupta et al. (ICML 2015, *Deep Learning with Limited Numerical Precision*); Wang et al. (NeurIPS 2018, *Training Deep Neural Networks with 8-bit Floating Point Numbers*).
- **Purpose:** Formulate unbiased Stochastic Rounding (SR), prove expected value preservation $\mathbb{E}[\operatorname{SR}(x)] = x$, eliminate the memory-expensive FP32 master weight copy, and implement Kahan compensated accumulation.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [05. Block Floating Point BFP Microscaling MXFP and NVFP4](05_block_floating_point_bfp_microscaling_mxfp_and_nvfp4.ipynb) $\to$ `🔗 Downstream Syntheses:` [01. INT UINT Fixed Point Q-Format Dyadic and RNS Arithmetic](../14_quantisation/01_foundations_formats_and_observers/01_int_uint_fixed_point_q_format_dyadic_and_rns_arithmetic.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do NOT use Stochastic Rounding during inference or deterministic evaluation (non-deterministic noise breaks exact reproducibility of benchmark metrics); do NOT use low-period pseudo-random generators (correlations in noise introduce systematic gradient drift).

## Mathematical Foundations & Stochastic Rounding Calculus

1. **Stochastic Rounding Definition**:
   $$\operatorname{SR}(x) = \begin{cases} x_R, & \text{with probability } P = \frac{x - x_L}{x_R - x_L} = \frac{x - x_L}{\text{ULP}} \\ x_L, & \text{with probability } 1 - P = \frac{x_R - x}{\text{ULP}} \end{cases}$$
2. **Unbiased Expectation Invariant**:
   $$\mathbb{E}[\operatorname{SR}(x)] = x_R \left(\frac{x - x_L}{\text{ULP}}\right) + x_L \left(\frac{x_R - x}{\text{ULP}}\right) = x$$
3. **Master-Weight-Free Optimization**:
   $$W_{t+1}^{\text{BF16}} = \operatorname{SR}\left(W_t^{\text{BF16}} - \eta g_t\right) \implies \text{Zero FP32 master weight memory overhead}$$
4. **Kahan Compensated Summation**:
   $$y = v - c, \quad t = \text{accum} + y, \quad c_{\text{new}} = (t - \text{accum}) - y, \quad \text{accum}_{\text{new}} = t$$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
if str(r) not in sys.path: sys.path.insert(0, str(r))
w_path = r / "weights/13_precision_and_numerics/06_stochastic_rounding_and_master_weight_free_bf16_training.safetensors"
weights = load_file(str(w_path))
print(f"Loaded {len(weights)} tensors from {w_path.name}")

In [ ]:
def linear_anchor(x, weight, bias):
    return x @ weight + bias

def stochastic_rounding_bf16(x, noise):
    x_abs = torch.abs(x)
    is_zero = (x_abs == 0.0)
    x_safe = torch.where(is_zero, torch.ones_like(x_abs), x_abs)
    e = torch.floor(torch.log2(x_safe))
    ulp = 2.0 ** (e - 7.0)
    step_ratio = x_safe / ulp
    floor_ratio = torch.floor(step_ratio)
    frac = step_ratio - floor_ratio
    round_up = (noise < frac)
    q_mag = torch.where(round_up, (floor_ratio + 1.0) * ulp, floor_ratio * ulp)
    return torch.where(is_zero, torch.zeros_like(x), torch.sign(x) * q_mag)

def deterministic_rtne_bf16(x):
    x_abs = torch.abs(x)
    is_zero = (x_abs == 0.0)
    x_safe = torch.where(is_zero, torch.ones_like(x_abs), x_abs)
    e = torch.floor(torch.log2(x_safe))
    ulp = 2.0 ** (e - 7.0)
    step_ratio = x_safe / ulp
    q_mag = torch.round(step_ratio) * ulp
    return torch.where(is_zero, torch.zeros_like(x), torch.sign(x) * q_mag)

def master_weight_free_sgd_step(w_bf16, grad, lr, noise, use_stochastic_rounding=True):
    target_w = w_bf16 - float(lr) * grad
    if use_stochastic_rounding:
        return stochastic_rounding_bf16(target_w, noise)
    return deterministic_rtne_bf16(target_w)

def kahan_accumulate_step(accum, val, comp):
    y = val - comp
    t = accum + y
    comp_new = (t - accum) - y
    return t, comp_new

In [ ]:
# 1. Linear anchor parity
y_anchor = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
assert torch.max(torch.abs(y_anchor - weights["test_output"])) < 1e-6

# 2. Stochastic Rounding & RTNE Parity
sr_out = stochastic_rounding_bf16(weights["x_float"], weights["noise"])
assert torch.max(torch.abs(sr_out - weights["sr_out_ref"])) < 1e-5
rtne_out = deterministic_rtne_bf16(weights["x_float"])
assert torch.max(torch.abs(rtne_out - weights["rtne_out_ref"])) < 1e-5

# 3. Master-Weight-Free SGD Step Parity
sgd_sr = master_weight_free_sgd_step(weights["w_init"], weights["grad_tiny"], lr=0.01, noise=weights["noise"], use_stochastic_rounding=True)
assert torch.max(torch.abs(sgd_sr - weights["sgd_sr_ref"])) < 1e-5
sgd_rtne = master_weight_free_sgd_step(weights["w_init"], weights["grad_tiny"], lr=0.01, noise=weights["noise"], use_stochastic_rounding=False)
assert torch.max(torch.abs(sgd_rtne - weights["sgd_rtne_ref"])) < 1e-5

# 4. Kahan Summation Parity
k_acc, k_c = kahan_accumulate_step(weights["k_accum"], weights["k_val"], weights["k_comp"])
assert torch.max(torch.abs(k_acc - weights["k_accum_ref"])) < 1e-5
assert torch.max(torch.abs(k_c - weights["k_comp_ref"])) < 1e-5
print("All Stochastic Rounding, SGD, and Kahan parity assertions verified!")

In [ ]:
# Microbenchmark: RTNE vs Stochastic Rounding Throughput
x_bench = weights["x_float"].repeat(100)
noise_bench = weights["noise"].repeat(100)

t0 = time.perf_counter()
for _ in range(500):
    _ = deterministic_rtne_bf16(x_bench)
rtne_us = (time.perf_counter() - t0) / 500 * 1e6
print(f"RTNE conversion latency: {rtne_us:.2f} us")

t0 = time.perf_counter()
for _ in range(500):
    _ = stochastic_rounding_bf16(x_bench, noise_bench)
sr_us = (time.perf_counter() - t0) / 500 * 1e6
print(f"Stochastic Rounding latency: {sr_us:.2f} us")

In [ ]:
# Pathological Misuse: Vanishing Updates under RTNE vs Stochastic Rounding Drift
w_rtne = torch.tensor(1.0)
w_sr = torch.tensor(1.0)
delta = torch.tensor(0.0002)  # Much smaller than ULP(1.0) = 0.0078125
n_steps = 1000

torch.manual_seed(42)
for _ in range(n_steps):
    # RTNE stalls: 1.0 - 0.0002 rounds back to 1.0 every step!
    w_rtne = deterministic_rtne_bf16(w_rtne - delta)
    # Stochastic rounding flips LSB probabilistically: P(flip) = 0.0002 / 0.0078125 ~ 0.0256
    noise_step = torch.rand(1)
    w_sr = stochastic_rounding_bf16(w_sr - delta, noise_step)

expected_target = 1.0 - (0.0002 * n_steps)  # 0.8
print(f"Target after 1000 steps: {expected_target:.4f}")
print(f"RTNE weight (Stalled!):  {float(w_rtne):.4f}")
print(f"SR weight (Drifted!):    {float(w_sr):.4f}")

assert float(w_rtne) == 1.0, "RTNE must completely stall on small updates!"
assert abs(float(w_sr) - expected_target) < 0.05, "Stochastic Rounding must track true expectation!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Unbiased Expectation**: Stochastic Rounding eliminates systematic directional truncation bias, guaranteeing $\mathbb{E}[\operatorname{SR}(x)] = x$ for arbitrary floating point bit widths.
2. **Master-Weight Elimination**: Directly updating BF16 parameters with Stochastic Rounding cuts parameter memory footprint by $50\%$, unlocking larger model batch sizes on memory-constrained GPUs.
3. **Zero-Overhead Silicon Implementation**: In hardware accelerators, Stochastic Rounding is realized with a simple 16-bit integer PRNG addition to the lower FP32 mantissa prior to truncation.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [06_stochastic_rounding_and_master_weight_free_bf16_training.md](../../markdown/13_precision_and_numerics/06_stochastic_rounding_and_master_weight_free_bf16_training.md)
- **PyTorch 2.x Module:** [06_stochastic_rounding_and_master_weight_free_bf16_training.py](../../pytorch2/13_precision_and_numerics/06_stochastic_rounding_and_master_weight_free_bf16_training.py)
- **Functional JAX Module:** [06_stochastic_rounding_and_master_weight_free_bf16_training.py](../../jax/13_precision_and_numerics/06_stochastic_rounding_and_master_weight_free_bf16_training.py)
- **Native Rust Candle Track:** [06_stochastic_rounding_and_master_weight_free_bf16_training.rs](../../candle/13_precision_and_numerics/06_stochastic_rounding_and_master_weight_free_bf16_training.rs)
- **Hardware Kernel (CPU / WGSL):** [06_stochastic_rounding_and_master_weight_free_bf16_training.rs](../../kernels/13_precision_and_numerics/06_stochastic_rounding_and_master_weight_free_bf16_training.rs)
- **Golden SafeTensors:** [06_stochastic_rounding_and_master_weight_free_bf16_training.safetensors](../../weights/13_precision_and_numerics/06_stochastic_rounding_and_master_weight_free_bf16_training.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 05. Block Floating Point BFP Microscaling MXFP and NVFP4](05_block_floating_point_bfp_microscaling_mxfp_and_nvfp4.ipynb) | [Master Index](../../README.md#13_precision_and_numerics) | [Next: 01. INT UINT Fixed Point Q-Format Dyadic and RNS Arithmetic](../14_quantisation/01_foundations_formats_and_observers/01_int_uint_fixed_point_q_format_dyadic_and_rns_arithmetic.ipynb) &rarr;